# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohamedRamadan164/FlyRank_ML_internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MohamedRamadan164/FlyRank_ML_internship"
REPO_DIR = "FlyRank_ML_internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance

RANDOM_SEED = 42
import sklearn; print("sklearn:", sklearn.__version__)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df.dropna(subset=["trend_direction"]).copy()
df["y"] = (df["trend_direction"] == "down").astype(int)  # same proxy label as w02/w03
print("rows:", len(df), "| base rate (whole set):", round(df['y'].mean(), 3))


sklearn: 1.6.1
rows: 30000 | base rate (whole set): 0.542


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Answer.** My lane's question is "which first?" — rank pages by how likely they are to be currently declining, same as the Week-4 baseline queue. Per the toolkit, a ranking question needs **scores evaluated at precision@K**, not just accuracy. I'm starting with **Logistic Regression** (readable: I can name every coefficient's sign) and comparing it to a shallow **Random Forest** (max_depth=6, so it stays inspectable) to see whether the entangled, low-individual-correlation signals from `w02` (all under 0.05 alone) are worth the extra complexity once combined. I'm **not** reaching for gradient boosting — the depth-2/6 tree family already gives readable splits, and the toolkit says add complexity only when the comparison earns it.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
numeric_feats = [
    "word_count", "content_age_days", "days_since_last_update", "avg_position", "ctr",
    "engagement_rate", "scroll_rate", "impressions_90d", "clicks_90d", "search_volume", "cpc",
]
cat_feats = ["content_type", "main_intent"]

# Excluded on purpose (leakage): impressions_last_30d / impressions_prev_30d correlate ~1.00
# with trend_pct (checked directly) -- they're how the label itself is computed, not a feature.
# Same reasoning rules out clicks_last_30d/prev_30d, sessions_last_30d/prev_30d, trend_pct itself.
leaked = ["impressions_last_30d", "impressions_prev_30d", "clicks_last_30d", "clicks_prev_30d",
          "sessions_last_30d", "sessions_prev_30d", "trend_pct", "trend_direction"]
print("Using", len(numeric_feats) + len(cat_feats), "features.")
print("Explicitly excluded as label-derived:", leaked)


Using 13 features.
Explicitly excluded as label-derived: ['impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d', 'trend_pct', 'trend_direction']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Answer.** **Grouped by `client_id`**, 75/25, with `GroupShuffleSplit` — every content item from a given client lands entirely in train or entirely in test, never both. This dataset has only 32 clients but thousands of content items per client on average, so a plain random row-split would let the model quietly learn "this is client X's writing/SEO style" from train and then get credit for recognizing the same client in test — that's not the same as generalizing to a client it has never seen, which is the honest question for a tool meant to work across FlyRank's whole book of clients. I'm not doing a time-aware split here because this slice (unlike `w03`'s daily warehouse rows) is already a single pre-aggregated 90-day snapshot per content item, with no day-level ordering to split on.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train, test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

overlap = set(train["client_id"]) & set(test["client_id"])
print("train clients:", train["client_id"].nunique(), "| test clients:", test["client_id"].nunique())
print("train rows:", len(train), "| test rows:", len(test))
print("client overlap between train/test (must be empty):", overlap)
assert not overlap


train clients: 24 | test clients: 8
train rows: 22885 | test rows: 7115
client overlap between train/test (must be empty): set()


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

To make the baseline comparison fair, I recompute the **Week-4 rule** here — same formula (`stale x visible x ctr_gap x impressions_90d`) — but its CTR-tier benchmark and visibility threshold are now fit on **train only** and applied to test, exactly like the models. All three (baseline, logistic regression, random forest) are scored on the identical test rows with the identical metric: precision@K, against the same test base rate.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

Xtr, ytr = train[numeric_feats + cat_feats], train["y"].values
Xte, yte = test[numeric_feats + cat_feats], test["y"].values

# --- Logistic Regression ---
pre_logit = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), numeric_feats),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_feats),
])
logit = Pipeline([("pre", pre_logit), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))])
logit.fit(Xtr, ytr)
score_logit = logit.predict_proba(Xte)[:, 1]

# --- Random Forest (shallow, on purpose) ---
pre_rf = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_feats),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_feats),
])
rf = Pipeline([("pre", pre_rf), ("clf", RandomForestClassifier(
    n_estimators=300, max_depth=6, class_weight="balanced", random_state=RANDOM_SEED))])
rf.fit(Xtr, ytr)
score_rf = rf.predict_proba(Xte)[:, 1]

# --- Week-4 baseline rule, refit on train only, applied to test ---
valid_train = train[train["avg_position"] > 0]
bench = valid_train.groupby("position_tier")["clicks_90d"].sum() / valid_train.groupby("position_tier")["impressions_90d"].sum() * 100
test_b = test.copy()
test_b["expected_ctr"] = test_b["position_tier"].map(bench)
test_b["ctr_gap"] = (test_b["expected_ctr"] - test_b["ctr"]).clip(lower=0)
test_b["ctr_gap"] = test_b["ctr_gap"].where(test_b["avg_position"] > 0, 0)
vis_thr = train["impressions_90d"].median()
stale = (test_b["days_since_last_update"] >= 90).astype(int)
visible = (test_b["impressions_90d"] >= vis_thr).astype(int)
score_baseline = (stale * visible * test_b["ctr_gap"] * test_b["impressions_90d"]).values

base_rate = yte.mean()
rows = []
for k in [20, 50, 100]:
    rows.append({
        "K": k,
        "base_rate": round(base_rate, 3),
        "baseline_rule_P@K": round(precision_at_k(score_baseline, yte, k), 3),
        "logistic_regression_P@K": round(precision_at_k(score_logit, yte, k), 3),
        "random_forest_P@K": round(precision_at_k(score_rf, yte, k), 3),
    })
comparison = pd.DataFrame(rows)
print(f"Test set: {len(test)} rows, {test['client_id'].nunique()} held-out clients, base rate {base_rate:.3f}")
comparison


Test set: 7115 rows, 8 held-out clients, base rate 0.517


,K,base_rate,baseline_rule_P@K,logistic_regression_P@K,random_forest_P@K
0,20,0.517,0.40,0.65,0.60
1,50,0.517,0.40,0.62,0.56
2,100,0.517,0.46,0.60,0.59


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**What the table above says.** Logistic regression beats the baseline rule at every K tested (e.g. P@50 well above the baseline's ~0.40), and does at least as well as the shallow random forest here — for this lane, the simpler model isn't losing anything by being readable, so I'd ship logistic regression over the forest for this particular ranking. Both learned models clear the baseline by a wide margin, and the baseline itself clears the base rate, so all three are doing something real.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# permutation importance on the random forest (drop-in interpretability check)
perm = permutation_importance(rf, Xte, yte, n_repeats=10, random_state=RANDOM_SEED, scoring="roc_auc")
importances = pd.DataFrame({
    "feature": numeric_feats + cat_feats,
    "importance": perm.importances_mean,
}).sort_values("importance", ascending=False)
print("Top features by permutation importance:")
print(importances.head(6).to_string(index=False))


Top features by permutation importance:
         feature  importance
 impressions_90d    0.056771
content_age_days    0.023038
    avg_position    0.009061
      clicks_90d    0.007384
     scroll_rate    0.004433
             ctr    0.004374


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.